# Session 1: what a beam does with a load
**A. Real-world problem.** A plank across a stream bends when you stand on it. Two things
hold it up: the two banks. How much of your weight does each bank carry? Where inside the
plank is the bending worst? Engineers answer with three pictures: the *reactions* at the
supports, the *shear force diagram* and the *bending moment diagram*.

**Plan.** (1) find the reactions with two balance rules, (2) cut the beam at x and find the
internal shear V and moment M, (3) draw both diagrams, (4) see where M is largest.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: reactions from balance
A simply supported beam of span `L` rests on a support at x = 0 (A) and x = L (B). A downward
point load `P` acts at distance `a` from A (and `b = L - a` from B). The beam does not move,
so two things must balance:

    forces:   RA + RB = P
    turning about A:   RB * L = P * a     ->   RB = P * a / L,   RA = P * b / L

The support nearer the load carries more of it.

In [ ]:
def reactions(P, a, L):
    """Support reactions (RA, RB) in the unit of P, for a load P at distance a from A."""
    b = L - a
    return P * b / L, P * a / L

## Step 2: cut the beam and look inside
Cut the beam at x and keep the left piece. Its inside face must supply

* a vertical force **V(x)** (shear force): V = RA, minus P once the cut is past the load;
* a turning effect **M(x)** (bending moment): M = RA * x, minus P * (x - a) past the load.

In [ ]:
def shear(x, P, a, L):
    """Shear force V(x), same unit as P."""
    ra, _ = reactions(P, a, L)
    x = np.asarray(x, dtype=float)
    return np.where(x < a, ra, ra - P)


def moment(x, P, a, L):
    """Bending moment M(x), unit of P times unit of length (sagging positive)."""
    ra, _ = reactions(P, a, L)
    x = np.asarray(x, dtype=float)
    return np.where(x < a, ra * x, ra * x - P * (x - a))

## Step 3: where is the bending worst?
M rises with slope RA, peaks under the load, then falls with slope -RB. So

    M_max = RA * a = P * a * b / L,   and for a centred load  M_max = P * L / 4.

V changes sign exactly where M peaks: slope of M = V.

In [ ]:
def max_moment(P, a, L):
    return P * a * (L - a) / L

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    P, L = 12.0, 8.0                       # kN, m
    fig, axes = plt.subplots(3, 1, figsize=(7, 7), sharex=True)
    x = np.linspace(0, L, 601)
    for a, c in ((2.0, "tab:blue"), (4.0, "tab:orange")):
        ra, rb = reactions(P, a, L)
        axes[0].plot(x, shear(x, P, a, L), c, label=f"load at {a:g} m: RA={ra:.1f}, RB={rb:.1f} kN")
        axes[1].plot(x, moment(x, P, a, L), c, label=f"load at {a:g} m: M max {max_moment(P, a, L):.1f} kN m")
        print(f"a={a}: RA={ra:.2f} kN, RB={rb:.2f} kN, Mmax={max_moment(P, a, L):.2f} kN m")
    axes[0].set_ylabel("shear force V (kN)")
    axes[1].set_ylabel("bending moment M (kN m)")
    for ax in axes[:2]:
        ax.axhline(0, color="k", lw=0.8)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    axes[0].set_title("Shear and moment, simply supported beam, P = 12 kN, L = 8 m")
    # the beam drawing
    axes[2].plot([0, L], [0, 0], "k", lw=6)
    for a, c in ((2.0, "tab:blue"), (4.0, "tab:orange")):
        axes[2].annotate("", xy=(a, 0.1), xytext=(a, 1.0), arrowprops=dict(color=c, width=2))
    axes[2].set_ylim(-0.5, 1.2)
    axes[2].set_yticks([])
    axes[2].set_xlabel("position along the beam x (m)")
    axes[2].set_title("The two load positions (arrows)", fontsize=9)
    fig.tight_layout()
    fig.savefig(OUT / "shear_moment.png", dpi=120)

    pos = np.linspace(0, L, 121)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(pos, max_moment(P, pos, L), label="M max = P a b / L")
    ax2.plot(L / 2, P * L / 4, "ro", label="centre load: P L / 4 = 24 kN m")
    ax2.set_xlabel("load position a from support A (m)")
    ax2.set_ylabel("largest bending moment (kN m)")
    ax2.set_title("The worst place for a load is the middle")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "moment_vs_position.png", dpi=120)
    print("centre load Mmax =", max_moment(P, L / 2, L), "kN m")

## Where this stops
This session says how big the forces inside the beam are, not whether the beam is strong or
stiff enough. For that we need the cross-section and the material: Session 2.

## Coding tasks
1. **Easy.** In `lesson.py`, change the load position to 5 m on the 8 m beam. Predict RA and RB first, then run it.
2. **Stretch.** Write `reactions_two_loads(P1, a1, P2, a2, L)` that adds the reactions from two separate point loads, and check that RA + RB = P1 + P2.
3. **Explain.** Explain, using the moment diagram, why a plank is most likely to break under your feet rather than at the bank.

## Assessment
- **A. Conceptual.** Why does the support closer to the load carry more of it? What happens to the two reactions when the load slides to exactly the middle?
- **B. Numerical.** A 20 kN load acts 1.5 m from the left support of a 5 m beam. Find RA, RB and the largest bending moment, with units, and check that the answers are sensible.
- **C. Graph and simulation interpretation.** On the moment graph, where is the peak, what is the slope of the line to its left, and what does that slope equal on the shear graph?
- **D. Design and optimisation challenge.** A shelf plank can take at most 15 kN m of bending moment. The load is 10 kN on a 6 m span. In which positions along the span may the load stand, if any?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    ra, rb = L.reactions(12.0, 2.0, 8.0)
    checks.append((f"worked example: P=12 kN at a=2 m, L=8 m gives RA=9 and RB=3 kN (got {ra:.2f}, {rb:.2f})",
                   abs(ra - 9) < 1e-9 and abs(rb - 3) < 1e-9))
    checks.append(("reactions add up to the load", abs(sum(L.reactions(7.5, 1.3, 5.0)) - 7.5) < 1e-9))
    checks.append(("worked example: M max = P a b / L = 18 kN m", abs(L.max_moment(12.0, 2.0, 8.0) - 18.0) < 1e-9))
    x = np.linspace(0, 8, 8001)
    m = L.moment(x, 12.0, 2.0, 8.0)
    checks.append(("moment is zero at both supports", abs(m[0]) < 1e-9 and abs(m[-1]) < 1e-9))
    checks.append(("largest moment of the diagram equals the formula", abs(m.max() - 18.0) < 1e-3))
    v = L.shear(x, 12.0, 2.0, 8.0)
    checks.append(("shear is +9 before the load and -3 after it", abs(v[0] - 9) < 1e-9 and abs(v[-1] + 3) < 1e-9))
    # hand check: slope of M equals V
    slope = np.gradient(m, x)
    checks.append(("slope of the moment diagram equals the shear force", abs(slope[1000] - 9) < 1e-6 and abs(slope[6000] + 3) < 1e-6))
    # edge: load on a support goes straight into it and bends nothing
    ra0, rb0 = L.reactions(10.0, 0.0, 4.0)
    checks.append(("edge case: load directly on support A: RA = P, no bending", abs(ra0 - 10) < 1e-9 and L.max_moment(10.0, 0.0, 4.0) == 0))
    best = max(range(0, 801), key=lambda i: L.max_moment(12.0, i / 100.0, 8.0))
    checks.append(("worst load position is the centre", best == 400))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
